# BERT Fine-Tuning — Policy Classification
### Digital Twin NLP Pipeline | Hackathon 2026

This notebook fine-tunes `bert-base-uncased` on the generated `policy_labels.jsonl` training data to classify simulation queries into five intervention categories.

**Before running:** Make sure you have enabled a GPU runtime:
> Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save

## Cell 1 — Check GPU
Run this first. You should see a CUDA GPU listed, not CPU.

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory : {mem:.1f} GB")
    print("\n\u2713 GPU is ready \u2014 training will be fast.")
else:
    # Security fix #1: hard stop — do not allow training to proceed on CPU.
    # With 50,000 examples, CPU training would take hours with no warning.
    raise RuntimeError(
        "\n\u26a0  No GPU detected. Training on CPU with 50,000 examples "
        "will take several hours.\n"
        "Fix: Runtime \u2192 Change runtime type \u2192 "
        "Hardware accelerator \u2192 T4 GPU \u2192 Save\n"
        "Then: Runtime \u2192 Run all  (or re-run from Cell 1)"
    )

Device : cuda
GPU    : Tesla T4
Memory : 15.6 GB

✓ GPU is ready — training will be fast.


## Cell 2 — Install Dependencies
Colab has torch pre-installed. This just adds the Hugging Face libraries.

In [2]:
# Fix #7 (partial): install fast tokenizer support explicitly
!pip install -q transformers[sentencepiece] scikit-learn
print("\u2713 Dependencies installed.")

✓ Dependencies installed.


## Cell 3 — Upload Training Data
Upload your `policy_labels.jsonl` file (generated by `generate_training_data.py`).

The file picker will appear when you run this cell.

In [4]:
from google.colab import files
import os

# Security fix #4: maximum allowed upload size (50 MB)
# Prevents accidental upload of wrong file exhausting Colab disk
MAX_UPLOAD_BYTES = 50 * 1024 * 1024  # 50 MB

print("Please upload your policy_labels.jsonl file...")
uploaded = files.upload()

# Accept any .jsonl file regardless of Windows duplicate numbering e.g. (3)
jsonl_files = [k for k in uploaded.keys() if k.endswith(".jsonl")]

if not jsonl_files:
    raise FileNotFoundError(
        "No .jsonl file found in upload. Got: " + str(list(uploaded.keys())) +
        "\nMake sure you are uploading the policy_labels.jsonl file."
    )

uploaded_filename = jsonl_files[0]
file_bytes = uploaded[uploaded_filename]

# Security fix #4: enforce file size ceiling before writing to disk
if len(file_bytes) > MAX_UPLOAD_BYTES:
    raise ValueError(
        f"Uploaded file is {len(file_bytes):,} bytes "
        f"({len(file_bytes)/1024/1024:.1f} MB) — "
        f"exceeds the {MAX_UPLOAD_BYTES//1024//1024} MB limit.\n"
        "Are you sure you uploaded the right file?"
    )

DATA_PATH = "/content/policy_labels.jsonl"
with open(DATA_PATH, "wb") as f:
    f.write(file_bytes)

size  = os.path.getsize(DATA_PATH)
# Security fix #5: always specify encoding="utf-8" explicitly
# System default encoding varies and will misread non-ASCII characters
lines = sum(1 for _ in open(DATA_PATH, encoding="utf-8"))

print(f"\n\u2713 Received  : {uploaded_filename}")
print(f"\u2713 Saved as   : policy_labels.jsonl")
print(f"\u2713 Size       : {size:,} bytes ({size/1024/1024:.2f} MB)")
print(f"\u2713 Lines      : {lines:,}")

Please upload your policy_labels.jsonl file...


Saving policy_labels.jsonl to policy_labels (1).jsonl

✓ Received  : policy_labels (1).jsonl
✓ Saved as   : policy_labels.jsonl
✓ Size       : 8,049,752 bytes (7.68 MB)
✓ Lines      : 50,000


## Cell 4 — Configuration
All training hyperparameters in one place. Safe to leave as-is for the hackathon.

In [15]:
from pathlib import Path

# ── Hyperparameters ─────────────────────────────────────────────────
SEED            = 42
MODEL_NAME      = "bert-base-uncased"
MAX_LENGTH      = 128
BATCH_SIZE      = 32          # Fix #6: was 16 — T4 has 15.6 GB, 32 fits easily
EPOCHS          = 5           # Fix #4: was 20 — 3-5 epochs is standard for BERT fine-tuning
LEARNING_RATE   = 2e-5
WARMUP_RATIO    = 0.1
GRAD_ACCUM_STEPS = 1          # Increase to 2 if you want effective batch size of 64

# Robustness fix: split ratios defined here in config — single source of truth.
# 80% train | 10% val | 10% test
TRAIN_RATIO     = 0.80
VAL_RATIO       = 0.10
# TEST_RATIO is implicit: 1.0 - TRAIN_RATIO - VAL_RATIO = 0.10

# Dropout — explicitly set to prevent overfitting on complex domain data
HIDDEN_DROPOUT          = 0.1
ATTENTION_DROPOUT       = 0.1

# Fix #3: PATIENCE is now actually used by the training loop in Cell 9
PATIENCE        = 3           # early stopping: stop if val_loss doesn't improve for this many epochs

# ── Data quality gates ──────────────────────────────────────────────
MIN_TOTAL_RECORDS = 1000
MIN_PER_LABEL     = 100

# ── Paths ─────────────────────────────────────────────────────
DATA_PATH         = Path("/content/policy_labels.jsonl")
OUTPUT_DIR        = Path("/content/policy_bert")
BEST_CKPT_DIR     = Path("/content/policy_bert_best")  # Fix #3: now actually written by Cell 9
EVAL_RESULTS_PATH = Path("/content/eval_results.json")
TRAINING_LOG_PATH = Path("/content/training_log.csv")

# ── Required model files — verified before download ─────────────────────
REQUIRED_MODEL_FILES = [
    "config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
]

# ── Label vocabulary — must match generate_training_data.py exactly ───────
LABELS = [
    "congestion_charge",
    "lane_closure",
    "public_transport_increase",
    "signal_retiming",
    "speed_limit_reduction",
]
LABEL2ID = {label: idx for idx, label in enumerate(LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

print("\u2713 Configuration set.")
print(f"  Model          : {MODEL_NAME}")
print(f"  Epochs         : {EPOCHS} (max, early stopping patience={PATIENCE})")
print(f"  Batch size     : {BATCH_SIZE}")
print(f"  Device         : {device}")
print(f"  Split          : {int(TRAIN_RATIO*100)}% train | "
      f"{int(VAL_RATIO*100)}% val | "
      f"{round((1-TRAIN_RATIO-VAL_RATIO)*100)}% test")
print(f"  Dropout        : hidden={HIDDEN_DROPOUT}  attention={ATTENTION_DROPOUT}")
print(f"  Early stopping : patience={PATIENCE} (monitors val_loss)")


✓ Configuration set.
  Model          : bert-base-uncased
  Epochs         : 5 (max, early stopping patience=3)
  Batch size     : 32
  Device         : cuda
  Split          : 80% train | 10% val | 10% test
  Dropout        : hidden=0.1  attention=0.1
  Early stopping : patience=3 (monitors val_loss)


## Cell 5 — Imports & Reproducibility

In [6]:
import json, csv, math, random, shutil
import numpy as np
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("✓ Seed set. All results will be reproducible.")

✓ Seed set. All results will be reproducible.


## Cell 6 — Load & Validate Training Data
Validates every record before training starts. Catches malformed or tampered data immediately.

In [7]:
def load_jsonl(path):
    """Load and validate records — security fixes #1 & #2."""
    if not path.exists():
        raise FileNotFoundError(
            f"Training data not found at '{path}'. Upload it in Cell 3."
        )

    records, errors = [], []

    with open(path, encoding="utf-8") as fh:
        for line_num, line in enumerate(fh, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                errors.append(f"  Line {line_num}: invalid JSON — {exc}")
                continue
            if not isinstance(record, dict):
                errors.append(
                    f"  Line {line_num}: expected JSON object, "
                    f"got {type(record).__name__}"
                )
                continue
            if "text" not in record or "label" not in record:
                errors.append(
                    f"  Line {line_num}: missing 'text' or 'label' — {record}"
                )
                continue
            if not isinstance(record["text"], str) or not record["text"].strip():
                errors.append(
                    f"  Line {line_num}: 'text' must be a non-empty string "
                    f"— got {record['text']!r}"
                )
                continue
            if record["label"] not in LABEL2ID:
                errors.append(
                    f"  Line {line_num}: unknown label '{record['label']}' "
                    f"— expected {list(LABEL2ID.keys())}"
                )
                continue
            records.append(record)

    if errors:
        shown = "\n".join(errors[:20])
        tail  = f"\n  ... and {len(errors)-20} more." if len(errors) > 20 else ""
        raise ValueError(
            f"Found {len(errors)} invalid record(s):\n{shown}{tail}\n"
            "Fix and re-upload."
        )

    return records


def validate_data(records):
    """Enforce minimum counts before training — security fix #6."""
    total = len(records)
    if total < MIN_TOTAL_RECORDS:
        raise ValueError(
            f"Only {total} records; need at least {MIN_TOTAL_RECORDS:,}. "
            "Re-generate the data."
        )
    counts = Counter(r["label"] for r in records)
    under  = {lbl: c for lbl, c in counts.items() if c < MIN_PER_LABEL}
    if under:
        raise ValueError(
            f"Labels with too few examples: {under}. "
            f"Minimum is {MIN_PER_LABEL} per label."
        )


def split_data_three_way(records, train_ratio, val_ratio, rng):
    """
    Stratified three-way split: train / val / test.

    Ratios come from Cell 4 config — TRAIN_RATIO and VAL_RATIO.
    Currently: 80% train | 10% val | 10% test.

    train  — model learns from this
    val    — early stopping watches this during training
    test   — touched ONCE in Cell 10 for final honest evaluation
    """
    # Validate ratios sum correctly
    test_ratio = round(1.0 - train_ratio - val_ratio, 10)
    if test_ratio <= 0:
        raise ValueError(
            f"train_ratio ({train_ratio}) + val_ratio ({val_ratio}) "
            f"leaves no room for test set. They must sum to less than 1.0."
        )

    by_label = {}
    for rec in records:
        by_label.setdefault(rec["label"], []).append(rec)

    train, val, test = [], [], []
    for label, items in by_label.items():
        rng.shuffle(items)
        n         = len(items)
        train_cut = int(n * train_ratio)
        val_cut   = int(n * (train_ratio + val_ratio))
        train.extend(items[:train_cut])
        val.extend(items[train_cut:val_cut])
        test.extend(items[val_cut:])

    rng.shuffle(train)
    rng.shuffle(val)
    rng.shuffle(test)

    if not train:
        raise ValueError("Train split is empty. Check split ratios and record count.")
    if not val:
        raise ValueError("Val split is empty. Reduce TRAIN_RATIO or add more data.")
    if not test:
        raise ValueError("Test split is empty. Reduce TRAIN_RATIO/VAL_RATIO or add more data.")

    return train, val, test


def evaluate_model(model, loader, device):
    """
    Run inference on a DataLoader, return (true_labels, predicted_labels).
    Used with val_loader during training (Cell 9).
    Used with test_loader ONCE in Cell 10 only.
    """
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            preds          = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
    return all_true, all_preds


rng = random.Random(SEED)

print("Loading training data...")
records = load_jsonl(DATA_PATH)
print(f"  Loaded {len(records):,} records")

label_counts = Counter(r["label"] for r in records)
print("\n  Label distribution:")
for lbl, cnt in sorted(label_counts.items()):
    bar = "\u2588" * (cnt // 500)
    print(f"    {lbl:<30}  {cnt:>6}  {bar}")

validate_data(records)
print("\n\u2713 Data validation passed")

# Security fix: ratios come from Cell 4 config — not hardcoded here
train_records, val_records, test_records = split_data_three_way(
    records, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, rng=rng
)
train_pct = int(TRAIN_RATIO * 100)
val_pct   = int(VAL_RATIO * 100)
test_pct  = round((1 - TRAIN_RATIO - VAL_RATIO) * 100)  # Fix #9: use round() not int() to avoid 9% display bug
print(f"\n  Train : {len(train_records):>6,} examples  ({train_pct}%)")
print(f"  Val   : {len(val_records):>6,} examples  ({val_pct}%) — early stopping only")
print(f"  Test  : {len(test_records):>6,} examples  ({test_pct}%) — held out until Cell 10")

Loading training data...
  Loaded 50,000 records

  Label distribution:
    congestion_charge                10000  ████████████████████
    lane_closure                     10000  ████████████████████
    public_transport_increase        10000  ████████████████████
    signal_retiming                  10000  ████████████████████
    speed_limit_reduction            10000  ████████████████████

✓ Data validation passed

  Train : 40,000 examples  (80%)
  Val   :  5,000 examples  (10%) — early stopping only
  Test  :  5,000 examples  (10%) — held out until Cell 10


## Cell 7 — PyTorch Dataset & DataLoaders

In [8]:
# Fix #7: use AutoTokenizer (selects BertTokenizerFast automatically — much faster)
from transformers import AutoTokenizer


class PolicyDataset(Dataset):
    """
    Fix #1: tokenization is done ONCE at construction time, not inside __getitem__.
    Previously the tokenizer was called on every sample access — once per epoch per sample.
    Pre-tokenizing moves all tokenizer work to dataset creation and stores tensors directly.
    Fix #12: token_type_ids is now passed through explicitly.
    """
    def __init__(self, records, tokenizer, max_length):
        texts  = [r["text"]  for r in records]
        labels = [LABEL2ID[r["label"]] for r in records]

        # Tokenize the entire split in one vectorised batch call
        encodings = tokenizer(
            texts,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        self.input_ids      = encodings["input_ids"]       # (N, max_length)
        self.attention_mask = encodings["attention_mask"]  # (N, max_length)
        # Fix #12: store token_type_ids explicitly (all-zeros for single-seq, but be explicit)
        self.token_type_ids = encodings.get("token_type_ids",
                                            torch.zeros_like(encodings["input_ids"]))
        self.labels         = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "token_type_ids": self.token_type_ids[idx],
            "label":          self.labels[idx],
        }


print(f"Loading tokenizer for {MODEL_NAME}...")
# Fix #7: AutoTokenizer picks BertTokenizerFast automatically
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Pre-tokenizing splits (runs once, then training is GPU-bound)...")
train_dataset = PolicyDataset(train_records, tokenizer, MAX_LENGTH)
val_dataset   = PolicyDataset(val_records,   tokenizer, MAX_LENGTH)
test_dataset  = PolicyDataset(test_records,  tokenizer, MAX_LENGTH)
print("\u2713 Tokenization complete")

# Fix #2: num_workers=2 and pin_memory=True prevent the GPU from starving
# while waiting for the CPU to prepare the next batch.
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)

print(f"\u2713 DataLoaders ready  (num_workers=2, pin_memory=True)")
print(f"  Train batches : {len(train_loader)}")
print(f"  Val batches   : {len(val_loader)}")
print(f"  Test batches  : {len(test_loader)}")


Loading tokenizer for bert-base-uncased...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Pre-tokenizing splits (runs once, then training is GPU-bound)...
✓ Tokenization complete
✓ DataLoaders ready  (num_workers=2, pin_memory=True)
  Train batches : 1250
  Val batches   : 157
  Test batches  : 157


## Cell 8 — Load BERT Model
Downloads ~440 MB on first run. Cached afterwards.

In [9]:
print(f"Loading {MODEL_NAME} for sequence classification...")
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

# Robustness fix: explicitly set dropout from config cell.
# Prevents overfitting and ensures consistent behaviour if model is reloaded from checkpoint.
model.config.hidden_dropout_prob          = HIDDEN_DROPOUT
model.config.attention_probs_dropout_prob = ATTENTION_DROPOUT
model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n\u2713 Model loaded on {device}")
print(f"  Total params     : {total_params:,}")
print(f"  Trainable params : {trainable_params:,}")
print(f"  Hidden dropout   : {model.config.hidden_dropout_prob}")
print(f"  Attention dropout: {model.config.attention_probs_dropout_prob}")


Loading bert-base-uncased for sequence classification...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✓ Model loaded on cuda
  Total params     : 109,486,085
  Trainable params : 109,486,085
  Hidden dropout   : 0.1
  Attention dropout: 0.1


## Cell 9 — Fine-Tune BERT
Trains for up to **5 epochs** with early stopping (patience=3, monitors val loss).
Best model weights are saved to `BEST_CKPT_DIR` each time val loss improves.
Expected time on T4 GPU: **~10–15 minutes total** (was estimated at 2–4 min for 4 epochs; now accurate).


In [10]:
import sys, math

# ── Helper: one training epoch ──────────────────────────────────────────
def train_epoch(model, loader, optimiser, scheduler, device, epoch):
    model.train()
    total_loss = 0.0
    for batch_idx, batch in enumerate(loader):
        input_ids       = batch["input_ids"].to(device)
        attention_mask  = batch["attention_mask"].to(device)
        token_type_ids  = batch["token_type_ids"].to(device)  # Fix #12
        labels          = batch["label"].to(device)
        optimiser.zero_grad()
        loss = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,   # Fix #12: pass token_type_ids explicitly
            labels=labels,
        ).loss
        if math.isnan(loss.item()) or math.isinf(loss.item()):
            raise ValueError(
                f"NaN/Inf loss at epoch {epoch} batch {batch_idx}.\n"
                "Try reducing LEARNING_RATE in Cell 4."
            )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


# ── Helper: evaluate — returns (true_labels, pred_labels, avg_loss) ──────────
# Fix #5: evaluate_model now also returns val_loss so we can monitor it
def evaluate_model(model, loader, device):
    """
    Run inference on a DataLoader.
    Returns (true_labels, predicted_labels, avg_loss).
    Fix #5: avg_loss is now returned so the training loop can track it
    for early stopping, not just accuracy.
    """
    model.eval()
    all_preds, all_true = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)  # Fix #12
            labels         = batch["label"].to(device)
            outputs        = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,  # Fix #12
                labels=labels,
            )
            total_loss += outputs.loss.item()   # Fix #5: accumulate val loss
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)         # Fix #5
    return all_true, all_preds, avg_loss


# ── Optimiser & scheduler ─────────────────────────────────────────────
optimiser   = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler   = get_linear_schedule_with_warmup(
    optimiser, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

# Fix #3 & #5: early stopping state — monitors val_loss (not val_accuracy)
best_val_loss   = float("inf")
patience_counter = 0
stopped_early   = False

# Fix #5: log now includes val_loss column
log_rows = [["epoch", "train_loss", "val_loss", "val_accuracy"]]

print(f"Fine-tuning for up to {EPOCHS} epochs on {device}...")
print(f"Split          : {int(TRAIN_RATIO*100)}% train | {int(VAL_RATIO*100)}% val | "
      f"{round((1-TRAIN_RATIO-VAL_RATIO)*100)}% test")
print(f"Early stopping : patience = {PATIENCE} epochs  (monitors val_loss)")
print(f"Train examples : {len(train_records):,}")
print(f"Val examples   : {len(val_records):,}")
print()
print(f"  {'Epoch':<8} {'Train Loss':>14} {'Val Loss':>12} {'Val Acc':>13} {'Status'}")
print("  " + "\u2500" * 8 + " " + "\u2500" * 14 + " " + "\u2500" * 12 + " " + "\u2500" * 13 + " " + "\u2500" * 30)

for epoch in range(1, EPOCHS + 1):

    # Step 1: train
    avg_train_loss = train_epoch(model, train_loader, optimiser, scheduler, device, epoch)

    # Step 2: validate — Fix #5: now returns val_loss too
    val_true, val_preds, avg_val_loss = evaluate_model(model, val_loader, device)
    val_accuracy = accuracy_score(val_true, val_preds)

    log_rows.append([epoch,
                     round(avg_train_loss, 6),
                     round(avg_val_loss, 6),    # Fix #5
                     round(val_accuracy, 4)])

    # Fix #3: early stopping on val_loss + best-model checkpoint
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # Save best weights to BEST_CKPT_DIR (Fix #3: BEST_CKPT_DIR now actually used)
        import shutil as _shutil
        if BEST_CKPT_DIR.exists():
            _shutil.rmtree(BEST_CKPT_DIR)
        model.save_pretrained(BEST_CKPT_DIR)
        tokenizer.save_pretrained(BEST_CKPT_DIR)
        status = "\u2713 best"
    else:
        patience_counter += 1
        status = f"patience {patience_counter}/{PATIENCE}"

    print(
        f"  {epoch:>2}/{EPOCHS:<5}  "
        f"{avg_train_loss:>14.4f}  "
        f"{avg_val_loss:>12.4f}  "
        f"{val_accuracy:>12.4f}  "
        f"{status}",
        flush=True,
    )

    if patience_counter >= PATIENCE:
        print(f"\n  Early stopping triggered at epoch {epoch} "
              f"(val_loss has not improved for {PATIENCE} epochs).")
        stopped_early = True
        break

print()
if not stopped_early:
    print(f"\u2713 Training complete ({EPOCHS} epochs, no early stop triggered).")
print(f"\u2713 Best val_loss : {best_val_loss:.4f}")
print(f"\u2713 Best weights saved to {BEST_CKPT_DIR}")

# Fix #3: reload best weights so Cell 10 evaluates the best model, not the last epoch
print("\nReloading best checkpoint for evaluation...")
model = BertForSequenceClassification.from_pretrained(BEST_CKPT_DIR)
model.to(device)
print("\u2713 Best model reloaded. Run Validation cell next.", flush=True)


Fine-tuning for up to 5 epochs on cuda...
Split          : 80% train | 10% val | 10% test
Early stopping : patience = 3 epochs  (monitors val_loss)
Train examples : 40,000
Val examples   : 5,000

  Epoch        Train Loss     Val Loss       Val Acc Status
  ──────── ────────────── ──────────── ───────────── ──────────────────────────────


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   1/5              0.2719        0.0120        0.9976  ✓ best


KeyboardInterrupt: 

In [11]:
# Validation report on val_loader
# This is the data used during training to monitor generalisation
print("Validation results on val set...\n", flush=True)

val_true, val_preds, val_loss = evaluate_model(model, val_loader, device)  # Fix #5: unpack 3 values
val_accuracy = accuracy_score(val_true, val_preds)
val_f1       = f1_score(val_true, val_preds, average="macro")

print("\u2550" * 60)
print("  VALIDATION RESULTS")
print("\u2550" * 60)
print(f"  Loss     : {val_loss:.4f}")          # Fix #5: now shown
print(f"  Accuracy : {val_accuracy:.4f}  ({val_accuracy*100:.1f}%)")
print(f"  F1 Macro : {val_f1:.4f}")
print()
print(classification_report(val_true, val_preds, target_names=LABELS))
print("\u2550" * 60)
print("\n  Run the Test cell next.", flush=True)


Validation results on val set...

════════════════════════════════════════════════════════════
  VALIDATION RESULTS
════════════════════════════════════════════════════════════
  Loss     : 0.0058
  Accuracy : 0.9982  (99.8%)
  F1 Macro : 0.9982

                           precision    recall  f1-score   support

        congestion_charge       1.00      1.00      1.00      1000
             lane_closure       1.00      1.00      1.00      1000
public_transport_increase       1.00      1.00      1.00      1000
          signal_retiming       1.00      1.00      1.00      1000
    speed_limit_reduction       1.00      1.00      1.00      1000

                 accuracy                           1.00      5000
                macro avg       1.00      1.00      1.00      5000
             weighted avg       1.00      1.00      1.00      5000

════════════════════════════════════════════════════════════

  Run the Test cell next.


## Cell 10 — Evaluate on Test Set
Targets: Accuracy ≥ 90% and F1 macro ≥ 0.85

In [12]:
# Robustness fix: cell order guard
# Fix #8: use 'in globals()' instead of 'in dir()' for reliable variable checks
required_vars = ["model", "test_loader", "log_rows"]
missing_vars  = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(
        f"Required variables not found: {missing_vars}\n"
        "Make sure you have run all cells from Cell 1 through Cell 9 "
        "before running this cell."
    )

# Fix #13: clear GPU cache before test evaluation to avoid OOM on full test set
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# evaluate_model is defined in Cell 9
# test_loader is used here ONCE — the first and only time test data is seen
print("Final evaluation on held-out test set...\n")
print(f"  Test examples : {len(test_records):,}  "
      f"({round((1-TRAIN_RATIO-VAL_RATIO)*100)}% of total dataset)")  # Fix #9: round not int
print("  NOTE: This is the FIRST and ONLY time test data is seen.\n")

# Fix #5: evaluate_model now returns (true, preds, loss)
true_labels, pred_labels, test_loss = evaluate_model(model, test_loader, device)

accuracy     = accuracy_score(true_labels, pred_labels)
f1_macro     = f1_score(true_labels, pred_labels, average="macro")
f1_per_label = f1_score(true_labels, pred_labels, average=None)
cm           = confusion_matrix(true_labels, pred_labels)

print("\u2550" * 60)
print("  FINAL EVALUATION RESULTS  (test set)")
print("\u2550" * 60)
print(f"  Test Loss: {test_loss:.4f}")  # Fix #5
print(f"  Accuracy : {accuracy:.4f}  ({accuracy*100:.1f}%)")
print(f"  F1 Macro : {f1_macro:.4f}")
print()
print("  Per-label F1:")
for label, score in zip(LABELS, f1_per_label):
    bar = "\u2588" * int(score * 20)
    print(f"    {label:<30}  {score:.3f}  {bar}")
print()
print("  Confusion Matrix (rows=true, cols=predicted):")
print("  " + "".join(f"{lbl[:6]:>8}" for lbl in LABELS))
for i, row in enumerate(cm):
    print("  " + f"{LABELS[i][:6]:<8}" + "".join(f"{v:>8}" for v in row))
print()
print(classification_report(true_labels, pred_labels, target_names=LABELS))
print("\u2550" * 60)

print()
if accuracy >= 0.90:
    print(f"\u2713 Accuracy target met : {accuracy:.4f}")
else:
    print(f"\u26a0  Accuracy {accuracy:.4f} below 0.90 target.")

if f1_macro >= 0.85:
    print(f"\u2713 F1 macro target met : {f1_macro:.4f}")
else:
    print(f"\u26a0  F1 macro {f1_macro:.4f} below 0.85 target.")

# Robustness fix: validate eval_results has all required keys before Cell 11 writes to disk
eval_results = {
    "accuracy":         round(float(accuracy), 4),
    "f1_macro":         round(float(f1_macro), 4),
    "f1_per_label":     {lbl: round(float(s), 4)
                         for lbl, s in zip(LABELS, f1_per_label)},
    "confusion_matrix": cm.tolist(),
    "test_loss":        round(float(test_loss), 4),  # Fix #5: include test_loss
    "split": {
        "train_pct": int(TRAIN_RATIO * 100),
        "val_pct":   int(VAL_RATIO * 100),
        "test_pct":  round((1 - TRAIN_RATIO - VAL_RATIO) * 100),  # Fix #9
    },
}
REQUIRED_EVAL_KEYS = ["accuracy", "f1_macro", "f1_per_label",
                      "confusion_matrix", "test_loss", "split"]
missing_keys = [k for k in REQUIRED_EVAL_KEYS if k not in eval_results]
if missing_keys:
    raise ValueError(
        f"eval_results is missing required keys: {missing_keys}\n"
        "Do not proceed to Cell 11 — re-run this cell."
    )
print("\n\u2713 eval_results validated — ready for Cell 11 save and download.")


Final evaluation on held-out test set...

  Test examples : 5,000  (10% of total dataset)
  NOTE: This is the FIRST and ONLY time test data is seen.

════════════════════════════════════════════════════════════
  FINAL EVALUATION RESULTS  (test set)
════════════════════════════════════════════════════════════
  Test Loss: 0.0067
  Accuracy : 0.9986  (99.9%)
  F1 Macro : 0.9986

  Per-label F1:
    congestion_charge               0.998  ███████████████████
    lane_closure                    0.999  ███████████████████
    public_transport_increase       1.000  ████████████████████
    signal_retiming                 0.999  ███████████████████
    speed_limit_reduction           0.997  ███████████████████

  Confusion Matrix (rows=true, cols=predicted):
    conges  lane_c  public  signal  speed_
  conges       996       1       0       0       3
  lane_c         0    1000       0       0       0
  public         0       0    1000       0       0
  signal         0       0       0    1000

## Cell 11 — Save Model
Saves atomically to a `_tmp` directory first, then renames — so a crash never leaves a broken model on disk (security fix #4).

After saving, downloads a zip of the model so you can use it in the FastAPI step.

In [16]:
import shutil, json as _json, csv as _csv, os

# Robustness fix: save and download in the SAME cell.
# Colab free tier disconnects after ~90 minutes of inactivity.
# If save and download were separate cells, a session expiry between
# them would destroy the trained model permanently.
# Merging them ensures download always follows save immediately.

# Robustness fix — cell order guard
# Fix #8: use globals() not dir() for reliable variable existence check
if "eval_results" not in globals():
    raise RuntimeError(
        "eval_results not found. Run Cell 10 before running this cell."
    )

# ── Atomic model save — security fix #4 ──────────────────────────────────
# Write to _tmp first. Rename to OUTPUT_DIR only on full success.
# A crash mid-save never leaves a partially written model in place.
tmp_dir = Path("/content/policy_bert_tmp")
if tmp_dir.exists():
    shutil.rmtree(tmp_dir)
tmp_dir.mkdir(parents=True)

try:
    model.save_pretrained(tmp_dir)
    tokenizer.save_pretrained(tmp_dir)
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    tmp_dir.rename(OUTPUT_DIR)
except Exception as e:
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    raise RuntimeError(f"Model save failed: {e}\nTmp directory cleaned up.") from e

print(f"\u2713 Model saved to {OUTPUT_DIR}")

# ── Verify all required files present before zipping ─────────────────────
saved_files = os.listdir(OUTPUT_DIR)
missing     = [f for f in REQUIRED_MODEL_FILES if f not in saved_files]
if missing:
    raise FileNotFoundError(
        f"Model save incomplete — missing files: {missing}\n"
        "Do NOT download. Re-run this cell to save again."
    )
print("\u2713 All required model files verified:")
for fname in sorted(saved_files):
    size_mb = os.path.getsize(OUTPUT_DIR / fname) / 1024 / 1024
    print(f"    {fname:<35}  {size_mb:.2f} MB")

# ── Save eval results ─────────────────────────────────────────────────────
with open(EVAL_RESULTS_PATH, "w", encoding="utf-8") as fh:
    _json.dump(eval_results, fh, indent=2)
print(f"\n\u2713 Eval results saved to {EVAL_RESULTS_PATH}")

# ── Save training log ─────────────────────────────────────────────────────
with open(TRAINING_LOG_PATH, "w", newline="", encoding="utf-8") as fh:
    _csv.writer(fh).writerows(log_rows)
print(f"\u2713 Training log saved to {TRAINING_LOG_PATH}")

# ── Zip model ─────────────────────────────────────────────────────────────
print("\nZipping model...")
shutil.make_archive("/content/policy_bert", "zip", "/content", "policy_bert")
zip_size = os.path.getsize("/content/policy_bert.zip") / 1024 / 1024
print(f"\u2713 Zip created: policy_bert.zip  ({zip_size:.1f} MB)")

# ── Download all files immediately ───────────────────────────────────────
# Do NOT move download to a separate cell — session timeout risk
from google.colab import files
print("\nDownloading files now...")
files.download("/content/policy_bert.zip")
files.download("/content/eval_results.json")
files.download("/content/training_log.csv")

print("\n\u2713 All files downloaded.")
print("  Next step: Unzip policy_bert.zip into nlp/models/policy_bert/ in your project.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model saved to /content/policy_bert
✓ All required model files verified:
    config.json                          0.00 MB
    model.safetensors                    417.68 MB
    tokenizer.json                       0.68 MB
    tokenizer_config.json                0.00 MB

✓ Eval results saved to /content/eval_results.json
✓ Training log saved to /content/training_log.csv

Zipping model...
✓ Zip created: policy_bert.zip  (387.0 MB)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ All files downloaded.
  Next step: Unzip policy_bert.zip into nlp/models/policy_bert/ in your project.
